# Train the open people model (RF-DETR Medium, Apache 2.0) to replace the NAS model

1. Downloads dataset **version 3** of `players-detection-my09y` (once; kept in Drive afterwards).
2. Keeps only **goalkeeper / player / referee** (the ball has its own model), with the pipeline's class ids.
3. Trains **RF-DETR Medium** at 640 px on the T4 (about 2-4 h; early stopping). Checkpoints are copied
   to Drive every 10 minutes; after a disconnect, **Run all** again and training resumes.
4. Runs the model on both CVAT clips and saves its detections + speed for Claude to score.

**Runtime:** T4 GPU. Colab Secret `ROBOFLOW_API_KEY` is needed only the first time (to download the
dataset). **Runtime -> Run all**; do not edit cells. Results: `MyDrive/Playbook/people_model/`.
Using another Google account? Share `MyDrive/Playbook` with it and add a shortcut to `Playbook` in its My Drive.

In [ ]:
# Cell 1 — GPU check, Drive, settings
import os, json, time, glob, shutil, subprocess, sys
g = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True) if shutil.which('nvidia-smi') else None
if g is None or g.returncode != 0:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> T4 GPU, then re-run.')
print('GPU:', g.stdout.strip())
from google.colab import drive
drive.mount('/content/drive')
if not os.path.isdir('/content/drive/MyDrive/Playbook'):
    raise SystemExit('MyDrive/Playbook not found. Other Google account? Add a shortcut to the shared Playbook folder in My Drive.')
DEST = '/content/drive/MyDrive/Playbook/people_model'
OUT = f'{DEST}/train'
os.makedirs(OUT, exist_ok=True)
WORKSPACE, PROJECT, VERSION = 'muwafag-hussain-o1ghr', 'players-detection-my09y', 3
RES, EPOCHS, BATCH, ACCUM = 640, 60, 4, 4
CLIPS = {'hilal_hazm_B': 'HILAL-HAZM_match_B_up10.mp4', 'hilal_ahli_B': 'HILAL-AHLI_match_B-up8.mp4'}

In [ ]:
# Cell 2 — Our repo + the two CVAT clips, and RF-DETR
PB, BRANCH = '/content/playbook', 'claude/setup-gpu-video-testing-JhgUH'
if not os.path.isdir(PB + '/.git'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/MuwafagQ/Playbook-program.git', PB], check=True)
else:  # a session that already ran: always use the latest code
    subprocess.run(['git', 'fetch', '-q', '--depth', '1', 'origin', BRANCH], cwd=PB, check=True)
    subprocess.run(['git', 'reset', '-q', '--hard', 'FETCH_HEAD'], cwd=PB, check=True)
for m in [m for m in list(sys.modules) if m.split('.')[0] in ('vision', 'tools', 'core')]:
    del sys.modules[m]  # re-import our code after an update
os.chdir(PB); sys.path.insert(0, PB)
VID = '/content/videos'; os.makedirs(VID, exist_ok=True)
subprocess.run(['git', 'fetch', '-q', '--depth', '1', 'origin', 'final-deliverable'], cwd=PB, check=True)
for f in CLIPS.values():
    if not os.path.exists(f'{VID}/{f}'):
        open(f'{VID}/{f}', 'wb').write(subprocess.run(['git', 'show', f'FETCH_HEAD:{f}'], cwd=PB,
                                                      capture_output=True, check=True).stdout)
!pip install -q "rfdetr[train]==1.11.0" roboflow
print('ready')

In [ ]:
# Cell 3 — Dataset version 3 (downloaded once, kept in Drive) -> people-only copy
SRC_DRIVE, SRC, PEOPLE = f'{DEST}/v3_coco', '/content/v3', '/content/v3_people'
if not os.path.exists(f'{SRC_DRIVE}/train/_annotations.coco.json'):
    from google.colab import userdata
    from roboflow import Roboflow
    Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY')).workspace(WORKSPACE).project(PROJECT) \
        .version(VERSION).download('coco', location=SRC, overwrite=True)
    shutil.copytree(SRC, SRC_DRIVE, dirs_exist_ok=True)
elif not os.path.exists(f'{SRC}/train/_annotations.coco.json'):
    shutil.copytree(SRC_DRIVE, SRC, dirs_exist_ok=True)
from tools.people_dataset import people_dataset
stats = people_dataset(SRC, PEOPLE)
json.dump(stats, open(f'{DEST}/dataset_stats.json', 'w'), indent=1)
print(json.dumps(stats, indent=1))

In [ ]:
# Cell 4 — Train RF-DETR Medium (resumes from the Drive checkpoint; copies to Drive every 10 min)
import threading
from rfdetr import RFDETRMedium
LOCAL = '/content/train_out'
os.makedirs(LOCAL, exist_ok=True)
FINISHED = os.path.exists(f'{OUT}/checkpoint_best_total.pth')  # written only when training ends
if not FINISHED and os.path.exists(f'{OUT}/last.ckpt') and not os.path.exists(f'{LOCAL}/last.ckpt'):
    shutil.copy(f'{OUT}/last.ckpt', f'{LOCAL}/last.ckpt')
    print('Resuming from the checkpoint in Drive.')

def sync_to_drive():
    for p in glob.glob(f'{LOCAL}/**/*', recursive=True):
        if not os.path.isfile(p):
            continue
        dst = os.path.join(OUT, os.path.relpath(p, LOCAL))
        if os.path.exists(dst) and os.path.getmtime(dst) >= os.path.getmtime(p):
            continue
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(p, dst + '.tmp'); os.replace(dst + '.tmp', dst)

stop = threading.Event()
def loop():
    while not stop.wait(600):
        try:
            sync_to_drive()
        except Exception as e:
            print('Drive sync skipped:', e)
threading.Thread(target=loop, daemon=True).start()

if FINISHED:
    stop.set()
    print('Training already finished (model in Drive); skipping. Delete', OUT, 'to train again.')
else:
    t0 = time.time()
    try:
        model = RFDETRMedium()
        model.train(dataset_dir=PEOPLE, epochs=EPOCHS, batch_size=BATCH, grad_accum_steps=ACCUM, lr=1e-4,
                    resolution=RES, output_dir=LOCAL, early_stopping=True, checkpoint_interval=5,
                    resume=f'{LOCAL}/last.ckpt' if os.path.exists(f'{LOCAL}/last.ckpt') else None)
        print(f'Training done in {(time.time() - t0) / 3600:.1f} h')
    finally:
        stop.set()
        sync_to_drive()
        print('Checkpoints copied to Drive.')
print(sorted(os.path.basename(p) for p in glob.glob(f'{OUT}/*')))

In [ ]:
# Cell 5 — Detections on both CVAT clips (same preprocessing as the pipeline) + speed
import cv2, numpy as np, pandas as pd, torch
from vision.people_model import load_people_model
from vision.preprocess import enhance_frame
import supervision as sv
best = next(p for p in [f'{OUT}/checkpoint_best_total.pth', f'{OUT}/checkpoint_best_ema.pth',
                        f'{OUT}/checkpoint_best_regular.pth'] if os.path.exists(p))
print('weights:', best)
people = load_people_model(best)
speed = {}
for name, f in CLIPS.items():
    cap = cv2.VideoCapture(f'{VID}/{f}'); rows, i, times = [], 0, []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        img = enhance_frame(frame, enabled=True)
        torch.cuda.synchronize(); t = time.time()
        det = sv.Detections.from_inference(people.infer(img, confidence=0.10)[0])
        torch.cuda.synchronize(); times.append(time.time() - t)
        rows += [{'frame': i, 'x1': round(a), 'y1': round(b), 'x2': round(c), 'y2': round(d),
                  'conf': round(s, 3), 'class_id': int(k)}
                 for (a, b, c, d), s, k in zip(det.xyxy.tolist(), det.confidence.tolist(), det.class_id.tolist())]
        i += 1
    cap.release()
    pd.DataFrame(rows).to_csv(f'{DEST}/{name}_people_dets.csv', index=False)
    speed[name] = 1000 * float(np.median(times[5:]))
    print(f'{name}: {i} frames, {len(rows)} detections, {speed[name]:.1f} ms/frame')
json.dump({'ms_per_frame_median_T4': speed, 'resolution': RES, 'weights': os.path.basename(best)},
          open(f'{DEST}/people_speed.json', 'w'), indent=1)
print('\nDone. Tell Claude the results are in MyDrive/Playbook/people_model/.')